# CPW length sweep and de-embedding

Simulate 100, 400, and 800 µm CPW lines with both **lumped ports** and **wave ports** over 300 frequencies from 1 to 100 GHz. De-embed the 100 µm reference from each longer line and compare propagation, impedance, and distributed RLGC parameters.

Start with the basic [lumped-port](https://github.com/gdsfactory/gsim/blob/main/nbs/palace_cpw_lumped.ipynb) or [wave-port](https://github.com/gdsfactory/gsim/blob/main/nbs/palace_cpw_waveport.ipynb) example for a single simulation. This notebook is self-contained and runs both sweeps when executed from top to bottom.

**Requirements:**

- IHP PDK: `uv pip install ihp-gdsfactory`
- scikit-rf: `uv pip install scikit-rf`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation

## Define the CPW

In [ ]:
import gdsfactory as gf
from ihp import LAYER, PDK

PDK.activate()


@gf.cell
def gsg_electrode(
    length: float = 800,
    s_width: float = 20,
    g_width: float = 40,
    gap_width: float = 15,
    layer=LAYER.TopMetal2drawing,
) -> gf.Component:
    """
    Create a GSG (Ground-Signal-Ground) electrode.

    Args:
        length: horizontal length of the electrodes
        s_width: width of the signal (center) electrode
        g_width: width of the ground electrodes
        gap_width: gap between signal and ground electrodes
        layer: layer for the metal
    """
    c = gf.Component()

    r1 = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
    r1.move((0, (g_width + s_width) / 2 + gap_width))

    _r2 = c << gf.c.rectangle((length, s_width), centered=True, layer=layer)

    r3 = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
    r3.move((0, -(g_width + s_width) / 2 - gap_width))

    c.add_port(
        name="o1",
        center=(-length / 2, 0),
        width=s_width,
        orientation=180,
        port_type="electrical",
        layer=layer,
    )

    c.add_port(
        name="o2",
        center=(length / 2, 0),
        width=s_width,
        orientation=0,
        port_type="electrical",
        layer=layer,
    )

    return c


c = gsg_electrode()
cc = c.copy()
cc.draw_ports()
cc

## Configure both port types

Use the geometry, materials, airbox, and solver settings from the basic examples. Both ports are excited in separate solves to obtain all four S-parameters required for de-embedding.

The examples use different airbox margins for the two port types. The study checks length consistency within each setup; a physical comparison between port types also requires airbox and mesh convergence.

In [ ]:
from gsim.common.stack import get_stack
from gsim.palace import DrivenSim


def setup_sim(cell, port_type, *, solver_type="Default"):
    sim = DrivenSim()
    if port_type == "lumped":
        output_name = "lumped"
    elif port_type == "wave":
        output_name = f"waveport-{solver_type.lower()}"
    else:
        raise ValueError(f"Unknown port type: {port_type}")
    sim.set_output_dir(f"./palace-sim-cpw-{output_name}-{cell.name}")
    sim.set_geometry(cell)
    sim.set_stack(get_stack())

    if port_type == "lumped":
        sim.set_airbox(margin_x=50, margin_y=0, z_above=100, z_below=100)
        for port_name in ["o1", "o2"]:
            sim.add_cpw_port(
                port_name, layer="topmetal2", s_width=20, gap_width=15, excited=True
            )
    else:
        sim.set_airbox(margin_x=0.0, margin_y=50, z_above=100.0, z_below=100.0)
        for port_name in ["o1", "o2"]:
            sim.add_wave_port(
                port_name,
                layer="topmetal2",
                max_size=True,
                mode=1,
                excited=True,
                eigensolver_type=solver_type,
                eigensolver_tol=1e-6,
                eigensolver_ksp_tol=1e-8,
                eigensolver_max_size=40,
                eigensolver_verbose=0,
            )

    sim.set_driven(fmin=1e9, fmax=100e9, num_points=300)
    print(sim.validate_config())
    return sim

## Run the length sweep

Run all six cases, reusing matching cached results, including the 800 µm simulations from the basic notebooks. Submit the jobs without waiting, then collect their results in the same order.

Lumped-port validation reads the mesh once per gap surface: two ports × two gaps gives four reads before the cache lookup. `Extracting results.tar.gz...` means downloaded results are being unpacked, including on cache hits. Use `verbose="full"` instead of `"status"` if you need solver logs.

In [ ]:
lengths_um = (100, 400, 800)
port_types = ("lumped", "wave")
port_labels = {"lumped": "Lumped ports", "wave": "Wave ports"}
job_ids = {}

for port_type in port_types:
    for length_um in lengths_um:
        component = gsg_electrode(length=length_um)
        simulation = setup_sim(component, port_type)
        simulation.mesh(
            preset="default",
            refined_mesh_size=2.0,
            max_mesh_size=40.0,
            fmax=150e9,
            verbose=False,
        )
        job_ids[port_type, length_um] = simulation.run(
            wait=False, check_cache=True, verbose="status"
        )

In [ ]:
import gsim

sweep_results = gsim.wait_for_results(list(job_ids.values()), verbose="status")
results_by_case = dict(zip(job_ids, sweep_results, strict=True))

## Prepare the two-port networks

Check that every result contains the full S-matrix before converting it to scikit-rf.

**Impedance reference:** 50 Ω matches the configured lumped-port resistance. For wave ports, `to_skrf(z0=50.0)` only assigns 50 Ω to the power-normalized S-matrix; this notebook does not measure the mode's circuit impedance or renormalize from it. The wave-port impedance and RLGC are therefore equivalent-circuit values in that assigned reference. Assigning twice the reference impedance to the same S-matrix doubles $Z_c$ without changing the extracted index. A physical impedance comparison needs a consistent voltage/power definition; see [Palace's port normalization conventions](https://awslabs.github.io/palace/dev/reference/#Wave-ports).

In [ ]:
# De-embedding requires all four S-parameters, including the output reflection.
networks = {}
for case, result in results_by_case.items():
    port_names = result.port_names
    assert len(port_names) == 2, "Expected a two-port CPW result."
    expected_pairs = {
        (to_port, from_port) for to_port in port_names for from_port in port_names
    }
    assert set(result.keys()) == expected_pairs, (
        "Run both port excitations before de-embedding."
    )
    networks[case] = result.to_skrf(z0=50.0)
    networks[case].frequency.unit = "GHz"

## De-embed the 100 µm reference

Split the 100 µm line into mirrored fixtures using [IEEE P370 NZC 2x-thru](https://scikit-rf.readthedocs.io/en/latest/api/calibration/generated/skrf.calibration.deembedding.IEEEP370_SE_NZC_2xThru.html). Assuming the same end fixtures at every length, removing them from the longer simulations leaves **300 µm (400−100)** and **700 µm (800−100)** sections. These are inferred networks, not separately simulated lines.

For a uniform, symmetric line, use $Z_c=sqrt{B/C}$ with nonnegative real part and unwrap the phase of $e^{gammaell}=(A+D)/2+B/Z_c$. This keeps the propagation constant continuous when a longer line passes through 180° of electrical length. Reconstructing the original networks checks the fixture orientation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import speed_of_light
from skrf.calibration.deembedding import IEEEP370_SE_NZC_2xThru


def _positive_real_branch(z):
    return np.where(np.real(z) < 0, -z, z)


def extract_modal_parameters(net, length_m):
    """Return Zc, gamma and neff of a uniform symmetric 2-port from its ABCD.

    Branch-free: Zc = sqrt(B/C) on the passive branch determines
    sinh(gamma*l) = B/Zc, hence exp(gamma*l) = cosh + sinh, leaving only the
    2*pi of the complex log to unwrap. No eigenvalue selection is involved, so
    the beta*l = pi eigenvalue collision of a low-loss line never arises.
    """
    a = net.a
    A, B, C, D = a[:, 0, 0], a[:, 0, 1], a[:, 1, 0], a[:, 1, 1]

    zc = _positive_real_branch(np.sqrt(B / C))
    cosh_gl = (A + D) / 2
    sinh_gl = B / zc
    exp_gl = cosh_gl + sinh_gl

    gamma_l = np.log(np.abs(exp_gl)) + 1j * np.unwrap(np.angle(exp_gl))
    gamma = gamma_l / length_m

    neff = np.full(len(net), np.nan)
    nonzero = net.f != 0
    neff[nonzero] = (
        np.imag(gamma[nonzero]) * speed_of_light / (2 * np.pi * net.f[nonzero])
    )

    # Symmetry residual: a uniform symmetric line has A == D.
    symmetry_error = np.max(np.abs(A - D) / np.maximum(np.abs(A), 1e-30))

    return {
        "zc": zc,
        "gamma": gamma,
        "neff": neff,
        "symmetry_error": symmetry_error,
    }


def p370_extract(short_2xthru, long_fix_dut_fix, delta_length_m):
    """Extract the additional line length using a mirrored IEEE P370 split."""
    reference_z0 = float(np.real(np.median(short_2xthru.z0[:, 0])))
    deembedding = IEEEP370_SE_NZC_2xThru(
        dummy_2xthru=short_2xthru,
        z0=reference_z0,
        use_z_instead_ifft=True,
        name="100 um non-zero-length 2xThru",
    )
    dut = deembedding.deembed(long_fix_dut_fix)

    # Explicitly reconstruct both topologies to check fixture orientation.
    reconstructed_short = deembedding.s_side1 ** deembedding.s_side2.flipped()
    reconstructed_long = deembedding.s_side1**dut ** deembedding.s_side2.flipped()
    result = extract_modal_parameters(dut, delta_length_m)
    result.update(
        {
            "dut": dut,
            "deembedding": deembedding,
            "split_error": np.max(np.abs(reconstructed_short.s - short_2xthru.s)),
            "reembed_error": np.max(np.abs(reconstructed_long.s - long_fix_dut_fix.s)),
        }
    )
    return result

In [ ]:
raw_parameters = {}
deembedded = {}

for port_type in port_types:
    reference = networks[port_type, 100]
    for length_um in lengths_um:
        raw_parameters[port_type, length_um] = extract_modal_parameters(
            networks[port_type, length_um], length_m=length_um * 1e-6
        )
    for total_length_um in lengths_um[1:]:
        remaining_um = total_length_um - 100
        deembedded[port_type, remaining_um] = p370_extract(
            reference,
            networks[port_type, total_length_um],
            delta_length_m=remaining_um * 1e-6,
        )

## Compare de-embedded transmission

The remaining length still changes the S21 attenuation and phase, even when the sections have the same propagation constant and impedance.

In [ ]:
for port_type in port_types:
    plt.figure()
    for remaining_um in (300, 700):
        network = deembedded[port_type, remaining_um]["dut"]
        network.plot_s_db(1, 0, label=f"{remaining_um} µm ({remaining_um + 100}−100)")
    plt.title(port_labels[port_type])
    plt.xlabel("Frequency [GHz]")
    plt.ylabel("De-embedded S21 [dB]")
    plt.legend()
    plt.show()

## Compare index and impedance across lengths

At each frequency, the same uniform line should have the same $n_{mathrm{eff}}$ and $Z_c$ regardless of length. Raw curves include port and end effects; the two de-embedded curves should agree more closely. Both extractions share the same 100 µm reference, so agreement checks consistency rather than independent accuracy or mesh convergence.

Compare the lengths within each port type. The wave-port impedance uses the assigned 50 Ω reference described above and cannot yet be interpreted as a physical impedance comparison with the lumped-port result.

In [ ]:
for port_type in port_types:
    for parameter in ("neff", "zc"):
        plt.figure()
        for length_um in lengths_um:
            values = raw_parameters[port_type, length_um][parameter].real
            plt.plot(
                networks[port_type, length_um].f / 1e9,
                values,
                label=f"Raw {length_um} µm",
            )
        for remaining_um in (300, 700):
            parameters = deembedded[port_type, remaining_um]
            plt.plot(
                parameters["dut"].f / 1e9,
                parameters[parameter].real,
                "--",
                label=f"De-embedded {remaining_um} µm ({remaining_um + 100}−100)",
            )
        title = port_labels[port_type]
        if parameter == "neff":
            plt.ylabel(r"$n_{\mathrm{eff}}$")
        elif port_type == "wave":
            plt.ylabel(r"Apparent Re($Z_c$) [Ω]")
            title += " — assigned 50 Ω reference"
        else:
            plt.ylabel(r"Re($Z_c$) [Ω]")
        plt.title(title)
        plt.xlabel("Frequency [GHz]")
        plt.legend()
        plt.show()

## Distributed line parameters

Compare resistance, inductance, conductance, and capacitance per unit length from the two remaining sections: $R+jomega L=gamma Z_c$ and $G+jomega C=gamma/Z_c$.

Negative resistance or conductance would prevent interpreting the extraction as a passive uniform RLGC line, even if the S-parameters remain passive. Check port normalization, fixture assumptions, mesh and airbox convergence, and sweep accuracy before interpreting small losses. Wave-port RLGC uses the assigned 50 Ω reference.

In [ ]:
units = {"R": "Ω/m", "L": "µH/m", "G": "S/m", "C": "pF/m"}
for port_type in port_types:
    for parameter, unit in units.items():
        plt.figure()
        for remaining_um in (300, 700):
            parameters = deembedded[port_type, remaining_um]
            network = parameters["dut"]
            series_per_m = parameters["gamma"] * parameters["zc"]
            shunt_per_m = parameters["gamma"] / parameters["zc"]
            values = {
                "R": series_per_m.real,
                "L": series_per_m.imag / network.frequency.w * 1e6,
                "G": shunt_per_m.real,
                "C": shunt_per_m.imag / network.frequency.w * 1e12,
            }
            plt.plot(network.f / 1e9, values[parameter], label=f"{remaining_um} µm")
        title = port_labels[port_type]
        if port_type == "wave":
            title += " — assigned 50 Ω reference"
        plt.title(title)
        plt.xlabel("Frequency [GHz]")
        plt.ylabel(f"{parameter} [{unit}]")
        plt.legend()
        plt.show()

## Comparing wave-port eigensolvers

To compare backends supported by your Palace build, use `setup_sim(component, "wave", solver_type="SLEPc")` and repeat with `solver_type="ARPACK"`. Mesh and run each simulation using the settings above, then compare S11/S21 magnitude and phase at matching frequencies. Keep the geometry, mesh, sweep, tolerances, and subspace size the same. `solver_type="Default"` lets Palace select the backend; output directories are separated by backend and geometry.